# Study 909 — Preferred Reset Premium 🔧

**When rates ripped higher in 2022, fixed-rate preferreds fell like long bonds — did the
*variable-rate* ones, which reset their coupon, hold up and out-carry them?**

Traditional preferred stock pays a **fixed** perpetual coupon (long duration). **Variable /
fixed-to-floating** preferreds reset off a short-rate benchmark, so their duration is short
and their income rises with the front end. We race Invesco's **VRP** and Global X's **PFFV**
(variable) against **PFF / PGX / PGF** (fixed), all excess-of-cash (minus BIL T-bills),
2014-06-30 → 2026-06-30.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `15c5bee54e98`);
the live cells run the fast synthetic control. Short history: the whole thesis leans on the
single 2022 hiking cycle — named on the Signal axis.*


## 1. The idea in one picture

A fixed-rate preferred is a long-duration bond wearing an equity coupon: when rates jump, its price falls hard. A variable-rate preferred **resets** its coupon off short rates, so it barely re-prices for duration and its income *grows* as the front end rises. The bet: in a high-rate regime the variable sleeve out-carries the fixed one on a rate-adjusted basis.

In [1]:
R = dict(cy2022_var=-11.4, cy2022_fix=-19.5, era_high_spread=3.9, era_high_t=2.41,
         era_low_spread=-0.27, era_low_t=-0.28, spread_full=1.28, t_nw_full=1.44)
print('2022 (rates ripping higher):')
print('  fixed sleeve   %+.1f%%' % R['cy2022_fix'])
print('  variable sleeve%+.1f%%  <- reset the coupon, lost 8 pts less'
      % R['cy2022_var'])

2022 (rates ripping higher):
  fixed sleeve   -19.5%
  variable sleeve-11.4%  <- reset the coupon, lost 8 pts less


## 2. But it's a *regime* story, not a law

Split the tape at the 2022 hiking cycle. In the **low-rate** years the two sleeves are a coin-flip; the whole premium appears only in the **high-rate** regime.

In [2]:
print('low-rate  2014-21: (var-fix) spread %+.2f%%/yr  (NW t = %+.2f)  ~nothing'
      % (R['era_low_spread'], R['era_low_t']))
print('high-rate 2022-26: (var-fix) spread %+.2f%%/yr  (NW t = %+.2f)  <- fires'
      % (R['era_high_spread'], R['era_high_t']))
print('full sample     : (var-fix) spread %+.2f%%/yr  (NW t = %+.2f)  thin/insig.'
      % (R['spread_full'], R['t_nw_full']))

low-rate  2014-21: (var-fix) spread -0.27%/yr  (NW t = -0.28)  ~nothing
high-rate 2022-26: (var-fix) spread +3.90%/yr  (NW t = +2.41)  <- fires
full sample     : (var-fix) spread +1.28%/yr  (NW t = +1.44)  thin/insig.


## 3. Is the detector honest? A live synthetic control

Plant a regime-contingent reset premium in a seeded toy world (variable out-carries fixed only in the high-rate months) and check the detector recovers it — and stays silent on the null (no edge). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from pref_reset import data, strategy as st
planted = st.synthetic_detect(data.synthetic_world(edge=0.0030, seed=909))
null = st.synthetic_detect(data.synthetic_world(edge=0.0, dur_hit=0.0, seed=909))
print('planted: spread NW t = %+.2f  (high-regime %+.1f%%/yr, low-regime %+.1f%%/yr)'
      % (planted['t_nw'], planted['spread_high_ann_pct'], planted['spread_low_ann_pct']))
print('null   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])

planted: spread NW t = +3.11  (high-regime +17.7%/yr, low-regime -1.1%/yr)
null   : spread NW t = +0.21  (should be ~0)


## 4. The honest verdict

On the real tape the variable sleeve *does* out-carry the fixed one — but **only in the high-rate regime** (+3.90%/yr, NW *t* = +2.41), and it evaporates when rates are floored (-0.27%/yr). Over the **full** sample the edge is a thin +1.28%/yr with *t* = +1.44 and a bootstrap CI across zero. And you can't *time* it: naively holding variable (excess Sharpe +0.40) beats switching on a rising-rate signal (+0.26). **Signal: Mixed** (real but regime-contingent), **Tradability: Fragile** (real-but-thin; the bankable form is a structural tilt to variable-rate preferreds, not a timed trade).